In [43]:
%pip install pandas numpy scikit-learn

Note: you may need to restart the kernel to use updated packages.


In [44]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

In [45]:
# 1. Load data
# -----------------------
df = pd.read_csv("Dataset_14-day_AA_depression_symptoms_mood_and_PHQ-9.csv")
df.columns = df.columns.str.strip()

In [46]:
df.tail(5)

,Unnamed: 0,user_id,phq1,phq2,phq3,phq4,phq5,phq6,phq7,phq8,...,q14,q16,q46,q47,happiness.score,time,period.name,start.time,phq.day,id
16145,16146,185,1.0,1.0,0.0,1.0,0.0,1.0,1.0,0.0,...,NaN,NaN,NaN,NaN,3,2017-02-04 10:44:56,morning,2017-01-07 18:55:29,11.632975,185
16146,16147,185,1.0,1.0,0.0,1.0,0.0,1.0,1.0,0.0,...,NaN,NaN,NaN,NaN,3,2017-01-29 22:00:47,evening,2017-01-07 18:55:29,6.102315,185
16147,16148,185,1.0,1.0,0.0,1.0,0.0,1.0,1.0,0.0,...,NaN,NaN,NaN,0.0,3,2017-01-18 18:30:06,evening,2017-01-07 18:55:29,-5.043993,185
16148,16149,185,1.0,1.0,0.0,1.0,0.0,1.0,1.0,0.0,...,0.0,NaN,NaN,NaN,3,2017-01-22 10:36:42,morning,2017-01-07 18:55:29,-1.372743,185
16149,16150,185,1.0,1.0,0.0,1.0,0.0,1.0,1.0,0.0,...,NaN,NaN,NaN,NaN,3,2017-01-08 15:01:18,midday,2017-01-07 18:55:29,-15.188993,185


In [47]:
# PHQ questions (main depression features)
phq_cols = [f"phq{i}" for i in range(1, 10)]

# EMA daily questions (q columns)
q_cols = [
    "q1","q2","q3","q4","q5","q6","q7","q8","q9",
    "q10","q11","q12","q13","q14","q16","q46","q47"
]

keep_cols = [
    "user_id",
    "age",
    "sex",
    "happiness.score",
    "time",
    "period.name",
    "phq.day"
] + phq_cols + q_cols

df = df[keep_cols]

In [48]:
for col in phq_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")
    df[col] = df[col].fillna(df[col].median())


In [49]:
# PHQ total score
df["phq_total"] = df[phq_cols].sum(axis=1)

def label(x):
    if x <= 4:
        return 0
    elif x <= 9:
        return 1
    elif x <= 14:
        return 2
    else:
        return 3

df["target"] = df["phq_total"].apply(label)

# remove leakage
df.drop(columns=["phq_total"], inplace=True)

In [50]:

for col in q_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

    # answered or not feature
    df[col + "_answered"] = df[col].notna().astype(int)

    # fill NaN using median
    df[col] = df[col].fillna(df[col].median())

In [51]:

# happiness score
df["happiness.score"] = pd.to_numeric(
    df["happiness.score"],
    errors="coerce"
)
df["happiness.score"] = df["happiness.score"].fillna(
    df["happiness.score"].median()
)

# age
df["age"] = pd.to_numeric(df["age"], errors="coerce")
df["age"] = df["age"].fillna(df["age"].median())

# phq.day
df["phq.day"] = pd.to_numeric(df["phq.day"], errors="coerce")
df["phq.day"] = df["phq.day"].fillna(df["phq.day"].median())

In [52]:
 #6. ENCODE SEX

df["sex"] = df["sex"].fillna(df["sex"].mode()[0])

df["sex"] = df["sex"].map({
    "male": 0,
    "female": 1
})

df["sex"] = df["sex"].fillna(0)

In [53]:
# 7. TIME CONVERSION + SORTING
# =========================================

df["time"] = pd.to_datetime(df["time"], errors="coerce")

# VERY IMPORTANT for time series
df = df.sort_values(["user_id", "time"])

In [54]:
# Better time-based features
df["month"] = df["time"].dt.month
df["day"] = df["time"].dt.day
df["hour"] = df["time"].dt.hour
df["weekday"] = df["time"].dt.dayofweek

In [55]:
# Fill missing values
df["month"] = df["month"].fillna(df["month"].median())
df["day"] = df["day"].fillna(df["day"].median())
df["hour"] = df["hour"].fillna(df["hour"].median())
df["weekday"] = df["weekday"].fillna(df["weekday"].median())


In [56]:
# 10. ENCODE PERIOD.NAME
# =========================================

df = pd.get_dummies(
    df,
    columns=["period.name"],
    dtype=int
)


In [57]:
# 12. TRUE TIME-SERIES FEATURE
# (ONLY HAPPINESS)
# =====================================

df["prev_happiness"] = (
    df.groupby("user_id")["happiness.score"]
    .shift(1)
)

df["rolling_happiness"] = (
    df.groupby("user_id")["happiness.score"]
    .transform(
        lambda x: x.shift(1).rolling(
            window=3,
            min_periods=1
        ).mean()
    )
)

df["happiness_trend"] = (
    df.groupby("user_id")["happiness.score"]
    .diff()
)

df["time_diff"] = (
    df.groupby("user_id")["time"]
    .diff()
    .dt.total_seconds()
)



In [58]:
# 13. FILL NaNs
# =====================================

df["prev_happiness"] = df["prev_happiness"].fillna(
    df["happiness.score"]
)

df["rolling_happiness"] = df["rolling_happiness"].fillna(
    df["happiness.score"]
)

df["happiness_trend"] = df["happiness_trend"].fillna(0)

df["time_diff"] = df["time_diff"].fillna(
    df["time_diff"].median()
)


# =====================================
# 14. DROP ORIGINAL TIME
# =====================================

df.drop(columns=["time"], inplace=True)


# =====================================
# 15. FINAL X and y
# =====================================

X = df.drop(columns=["target", "user_id"])
y = df["target"]


In [60]:

df["age"].nunique()
df["sex"].nunique()

2

In [59]:
df.head(55)

,user_id,age,sex,happiness.score,phq.day,phq1,phq2,phq3,phq4,phq5,...,day,hour,weekday,period.name_evening,period.name_midday,period.name_morning,prev_happiness,rolling_happiness,happiness_trend,time_diff
43,1,29.0,1.0,1,-14.486204,3.0,3.0,3.0,3.0,2.0,...,9,7,0,0,0,1,1.0,1.000000,0.0,32249.0
42,1,29.0,1.0,0,-10.002766,3.0,3.0,3.0,3.0,2.0,...,13,18,4,1,0,0,1.0,1.000000,-1.0,387369.0
19,1,29.0,1.0,2,-9.160775,3.0,3.0,3.0,3.0,2.0,...,14,15,5,0,1,0,0.0,0.500000,2.0,72748.0
41,1,29.0,1.0,2,-8.900208,3.0,3.0,3.0,3.0,2.0,...,14,21,5,1,0,0,2.0,1.000000,0.0,22513.0
33,1,29.0,1.0,3,-8.289155,3.0,3.0,3.0,3.0,2.0,...,15,12,6,0,1,0,2.0,1.333333,1.0,52795.0
31,1,29.0,1.0,4,-7.817245,3.0,3.0,3.0,3.0,2.0,...,15,23,6,1,0,0,3.0,2.333333,1.0,40773.0
22,1,29.0,1.0,3,-7.267234,3.0,3.0,3.0,3.0,2.0,...,16,12,0,0,1,0,4.0,3.000000,-1.0,47521.0
32,1,29.0,1.0,3,-6.950417,3.0,3.0,3.0,3.0,2.0,...,16,20,0,1,0,0,3.0,3.333333,0.0,27373.0
13,1,29.0,1.0,2,-6.486991,3.0,3.0,3.0,3.0,2.0,...,17,7,1,0,0,1,3.0,3.333333,-1.0,40040.0
21,1,29.0,1.0,3,-5.812269,3.0,3.0,3.0,3.0,2.0,...,17,23,1,1,0,0,2.0,2.666667,1.0,58296.0


In [61]:
df["happiness.score"].value_counts()

happiness.score
2    6550
3    3696
1    3633
0    1900
4     371
Name: count, dtype: int64